# Milestone 2 

## RAG Pipeline: Semantic and Hybrid Retrieval

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv()

True

### Initialize LLM on Groq

In [6]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=os.getenv("GROQ_API_KEY")
)

#response = llm.invoke("What is a good garden hose?")   # quick smoke test
#print(response.content)

### Load FAISS and Embedding Model

In [7]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
import sys
sys.path.append('../src')

from session_helper import init_session

In [19]:
MODEL_NAME = "all-MiniLM-L6-v2"
FAISS_INDEX_PATH = "../data/processed/faiss_index.bin"
FAISS_META_PATH  = "../data/processed/faiss_meta.pkl"

model = SentenceTransformer(MODEL_NAME)
index = faiss.read_index(FAISS_INDEX_PATH)

with open(FAISS_META_PATH, 'rb') as f:
    metadata = pickle.load(f)

print(f"Index loaded: {index.ntotal:,} vectors")
print(f"Metadata loaded: {len(metadata):,} records")

/home/ubu/miniforge3/envs/amz/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Index loaded: 20,000 vectors
Metadata loaded: 20,000 records


## Test Semantic Retrieval

In [11]:
def retrieve(query, k=5):
    query_vec = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, k)
    return [metadata[i] for i in indices[0]]

docs = retrieve("garden hose 50ft")
for d in docs:
    print(d['title'])

150FT-Movable-Hose-Reel-Cart-Hose-Pipe-Trolley-Outdoor-Garden-Yard-Plant-Water
iustus Garden Hose Expandable 50 Feet Lightweight Kink Free Water Hose 9 Function Spray Hose Nozzle 3-Layer Latex with 3/4 Brass Fittings Easy Storage Flexible Garden Hose for Car Washing
3-time 50ft Green Expandable Garden Hose Strongest Expanding Water Hose with Lightweight Webbing Brass Fittings Connectors Tap to Pressure Washer 8 Patterns Hose Sprayer
75FT Garden Hose-Heavy Duty Strongest Expandable Magic Water Hose with Double Latex Core, Solid Brass Connector 10 Pattern Spray Nozzle(Black)
[2019 Model] 50ft Non-Kink Expandable Garden Hose, 8-Pattern Spray Nozzle Included, 3/4” Brass Fittings with Shutoff Valve, Best 50' Foot Garden Hose


## Build Context from Retrieved Docs

In [13]:
def build_context(docs):
    return "\n\n".join(
        f"Product: {doc.get('title', 'N/A')}\n"
        f"ASIN: {doc.get('parent_asin', 'N/A')}\n"
        f"Rating: {doc.get('average_rating', 'N/A')}/5\n"
        f"Price: ${doc.get('price', 'N/A')}\n"
        f"Store: {doc.get('store', 'N/A')}"
        for doc in docs
    )

context = build_context(retrieve("garden hose 50ft"))   # smoke test
print(context)

Product: 150FT-Movable-Hose-Reel-Cart-Hose-Pipe-Trolley-Outdoor-Garden-Yard-Plant-Water
ASIN: B01LWV4MYF
Rating: 1.0/5
Price: $nan
Store: Unknown

Product: iustus Garden Hose Expandable 50 Feet Lightweight Kink Free Water Hose 9 Function Spray Hose Nozzle 3-Layer Latex with 3/4 Brass Fittings Easy Storage Flexible Garden Hose for Car Washing
ASIN: B07SJDXBTK
Rating: 4.2/5
Price: $nan
Store: iustus

Product: 3-time 50ft Green Expandable Garden Hose Strongest Expanding Water Hose with Lightweight Webbing Brass Fittings Connectors Tap to Pressure Washer 8 Patterns Hose Sprayer
ASIN: B01N0QPJGV
Rating: 2.7/5
Price: $nan
Store: Podura

Product: 75FT Garden Hose-Heavy Duty Strongest Expandable Magic Water Hose with Double Latex Core, Solid Brass Connector 10 Pattern Spray Nozzle(Black)
ASIN: B07QK6XKQW
Rating: 3.8/5
Price: $nan
Store: Brightown

Product: [2019 Model] 50ft Non-Kink Expandable Garden Hose, 8-Pattern Spray Nozzle Included, 3/4” Brass Fittings with Shutoff Valve, Best 50' Foot G

## Prompt Template

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

SYSTEM_PROMPT = """You are a helpful Amazon shopping assistant specializing 
in patio, lawn and garden products. Answer the question using ONLY the 
provided product context. Be concise and cite product names when possible. 
If the context does not contain enough information, say so."""

prompt_template = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

## Full RAG pipeline with LCEL

In [24]:
import re

def strip_think_tag(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()      # regex function to strip the <think> tags

rag_chain = (
    {
        "context": RunnableLambda(lambda q: build_context(retrieve(q))),
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

answer = strip_think_tag(rag_chain.invoke("What is a good garden hose for 50ft?"))  # smoke test
print(answer)

For a 50ft garden hose, the **iustus Garden Hose Expandable 50 Feet** (Rating: 4.2/5) is a strong recommendation. It features a 9-function spray nozzle, 3-layer latex construction, and 3/4" brass fittings for durability, paired with a lightweight, kink-free design. Alternatively, the **ACSTEP 50FT Expandable Garden Hose** (Rating: 4.1/5) is another top choice with a 9-function nozzle, high-pressure resistance, and leakproof design. Both offer excellent ratings and functionality for 50ft needs.


## BM25 Retrieval

In [25]:
import sys
sys.path.append('../src')
import bm25

con = init_session()

def retrieve_bm25(query, k=5):
    results = bm25.query_k_highest(con, query, k)
    return results.to_dict(orient='records')

bm25_docs = retrieve_bm25("garden hose 50ft")     # smoke test
for d in bm25_docs:
    print(d['title'])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Nifty Grower Expandable Garden Hose 50FT - Hoses Expandable 50 FT Heavy Duty with Double Latex Core - 50 Foot Hose with Brass Fittings - Collapsible Hose 50FT - No-Kink Expanding Hose 50 FT
Nifty Grower Expandable Garden Hose 100FT - Hoses Expandable 100 FT Heavy Duty with Double Latex Core - 100 Foot Hose with Brass Fittings - Collapsible Hose 100FT - No-Kink Expanding Hose 100 FT
Garden Hose 50ft
KETTOYA 50FT Expandable Garden Hose, Flexible Water Hose with 10-Pattern Spray Nozzle, Leak-Proof Retractable Heavy Duty Hose, 5-layer Latex Core, Durable 3750D, 3/4 Solid Brass Connectors, No-Kink
Sunifier Expandable Garden Hose 15 FT 25FT Flexible Garden Hose 50 FT 100 FT Expandable Water Hose with Garden Hose Nozzle for Lawn, Patio, Garden (75 FT, Black)


## Hybrid Retriever — Combine BM25 and Semantic Results

In [27]:
def retrieve_hybrid(query, k=5):
    semantic_docs = retrieve(query, k=k)
    bm25_docs = retrieve_bm25(query, k=k)
    
    seen = set()                             # deduplicate by parent_asin => semantic results take priority
    combined = []
    for doc in semantic_docs + bm25_docs:
        asin = doc.get('parent_asin')
        if asin not in seen:
            seen.add(asin)
            combined.append(doc)
    
    return combined[:k]                             # return top k after deduplication

hybrid_docs = retrieve_hybrid("garden hose 50ft")    # test
for d in hybrid_docs:
    print(d['title'])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

150FT-Movable-Hose-Reel-Cart-Hose-Pipe-Trolley-Outdoor-Garden-Yard-Plant-Water
iustus Garden Hose Expandable 50 Feet Lightweight Kink Free Water Hose 9 Function Spray Hose Nozzle 3-Layer Latex with 3/4 Brass Fittings Easy Storage Flexible Garden Hose for Car Washing
3-time 50ft Green Expandable Garden Hose Strongest Expanding Water Hose with Lightweight Webbing Brass Fittings Connectors Tap to Pressure Washer 8 Patterns Hose Sprayer
75FT Garden Hose-Heavy Duty Strongest Expandable Magic Water Hose with Double Latex Core, Solid Brass Connector 10 Pattern Spray Nozzle(Black)
[2019 Model] 50ft Non-Kink Expandable Garden Hose, 8-Pattern Spray Nozzle Included, 3/4” Brass Fittings with Shutoff Valve, Best 50' Foot Garden Hose


## Hybrid RAG chain

In [28]:
hybrid_rag_chain = (
    {
        "context": RunnableLambda(lambda q: build_context(retrieve_hybrid(q))),
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

answer = strip_think_tag(hybrid_rag_chain.invoke("What is a good garden hose for 50ft?"))  # test
print(answer)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

For a 50-foot garden hose, the **iustus Garden Hose Expandable 50 Feet** (4.2/5) and **ACSTEP 50FT Expandable Garden Hose** (4.1/5) are top choices. The iustus model offers a 9-function nozzle, kink-free flexibility, and brass fittings, while ACSTEP provides high-pressure durability and a 9-function nozzle. The **Komnn- Expandable Flexible Garden Hose** (3.1/5) is another option but has a lower rating.
